In [ ]:
!pip install torch-geometric -q

import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, DataLoader
from nn_conv import NNConv_old

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.8 MB/s eta 0:00:00


device(type='cuda')

In [ ]:
CSV_PATH = "/content/new_master_panel_residual_load.csv"

NODE_COL = "node_id"
TIME_COL = "datetime"
LAT_COL  = "lat"
LON_COL  = "lon"

#normalize residual_load (target) per node
PER_NODE_NORM_COLUMNS = [
    "residual_load",
]

FEATURE_COLUMNS = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "is_weekend",
    "is_holiday",
    "daylight_hours",
    "temperature",
    "heating_degree",
    "cooling_degree",
    "solar_ghi",
    "wind_100m",
    #lat_norm, lon_norm appended automatically in Cell 4
]

ORIGINAL_TARGET_COL = "residual_load"
TARGET_COL = "target_residual_load_norm"

#graph construction
GRAPH_METHOD = "hybrid"     #radius + kNN fallback
K_NEIGHBORS  = 3
RADIUS_KM    = 850.0
MIN_K        = 3
MAX_K        = 12
RBF_SIGMAS   = (0.03, 0.06, 0.12, 0.24)  #on normalized distance

#training hyperparams
BASE_CONFIG = {
    "hidden_width": 96,
    "kernel_width": 256,
    "depth": 5,
    "lr": 0.0010163,
    "weight_decay": 7.822e-7,
    "batch_size": 8,
    "num_epochs": 100,
}


In [ ]:
import numpy as np
import torch

def build_graph_from_coords(
    node_df,
    method="hybrid",          # "radius", "knn", or "hybrid"
    k_neighbors=3,
    radius_km=850.0,
    min_k=3,
    max_k=12,
    rbf_sigmas=(0.03, 0.06, 0.12, 0.24),  #on normalized distance
):
    """
    Returns:
      node_index_map: dict node_id -> contiguous index
      pos_t: (N,2) normalized [lat_norm, lon_norm]
      edge_index: (2,E) long tensor
      edge_attr: (E, 3 + len(rbf_sigmas)) float tensor: [Δlat_norm, Δlon_norm, dist_norm, rbf...]
      lat_norm, lon_norm: arrays (N,)
    """
    node_ids = node_df.index.to_list()
    N = len(node_ids)
    node_index_map = {nid: i for i, nid in enumerate(node_ids)}

    lat = node_df["lat"].values.astype(float)
    lon = node_df["lon"].values.astype(float)

    #min-max normalize coords
    lat_norm = (lat - lat.min()) / (lat.max() - lat.min() + 1e-9)
    lon_norm = (lon - lon.min()) / (lon.max() - lon.min() + 1e-9)
    pos = np.stack([lat_norm, lon_norm], axis=1)

    #compute full geographic distance matrix via haversine
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371.0
        lat1 = np.radians(lat1); lon1 = np.radians(lon1)
        lat2 = np.radians(lat2); lon2 = np.radians(lon2)
        dlat = lat2 - lat1; dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    lat_mesh1, lat_mesh2 = np.meshgrid(lat, lat, indexing="ij")
    lon_mesh1, lon_mesh2 = np.meshgrid(lon, lon, indexing="ij")
    dist_km = haversine_distance(lat_mesh1, lon_mesh1, lat_mesh2, lon_mesh2)  # (N,N)

    rows, cols = [], []

    def add_undirected_edges(i, neigh_idx):
        for j in neigh_idx:
            rows.append(i); cols.append(j)
            rows.append(j); cols.append(i)

    if method == "knn":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf
            neigh_idx = np.argsort(d)[:k_neighbors]
            add_undirected_edges(i, neigh_idx)

    elif method == "radius":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf
            within = np.where(d <= radius_km)[0]

            if within.size > max_k:
                within = within[np.argsort(d[within])[:max_k]]

            if within.size < min_k:
                within = np.argsort(d)[:max(min_k, k_neighbors)]

            add_undirected_edges(i, within)

    elif method == "hybrid":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf

            within = np.where(d <= radius_km)[0]

            if within.size > max_k:
                within = within[np.argsort(d[within])[:max_k]]

            if within.size < min_k:
                need = max(min_k, k_neighbors)
                within = np.argsort(d)[:need]

            add_undirected_edges(i, within)

    else:
        raise ValueError(f"Unknown method={method}. Use 'knn', 'radius', or 'hybrid'.")

    edge_index = torch.tensor([rows, cols], dtype=torch.long)

    #edge attributes: lat_norm, lon_norm + dist_norm + RBFs
    pos_t = torch.tensor(pos, dtype=torch.float32)  # (N,2)
    row, col = edge_index[0], edge_index[1]
    rel = pos_t[col] - pos_t[row]                   # (E,2)
    dlat = rel[:, 0:1]
    dlon = rel[:, 1:2]

    dist_vals = dist_km[row.numpy(), col.numpy()]
    dist_norm = dist_vals / (dist_km.max() + 1e-9)
    dist_norm = torch.tensor(dist_norm, dtype=torch.float32).unsqueeze(-1)

    rbf_feats = []
    for s in rbf_sigmas:
        s = float(s)
        rbf_feats.append(torch.exp(-(dist_norm ** 2) / (2.0 * (s ** 2) + 1e-12)))
    rbf = torch.cat(rbf_feats, dim=-1) if len(rbf_feats) > 0 else None

    if rbf is not None:
        edge_attr = torch.cat([dlat, dlon, dist_norm, rbf], dim=-1)
    else:
        edge_attr = torch.cat([dlat, dlon, dist_norm], dim=-1)

    return node_index_map, pos_t, edge_index, edge_attr, lat_norm, lon_norm


In [ ]:
def load_and_preprocess_panel(csv_path):
    df = pd.read_csv(csv_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values([NODE_COL, TIME_COL])

    #treat missing weather as 0
    WEATHER_COLUMNS = ["solar_ghi", "wind_100m"]
    for wcol in WEATHER_COLUMNS:
        if wcol in df.columns:
            df[wcol] = df[wcol].fillna(0.0)

    #node metadata for graph
    node_df = (
        df[[NODE_COL, LAT_COL, LON_COL]]
        .drop_duplicates(NODE_COL)
        .set_index(NODE_COL)
        .sort_index()
    )

    (node_index_map, pos_tensor, edge_index, edge_attr,
     lat_norm, lon_norm) = build_graph_from_coords(
        node_df,
        method=GRAPH_METHOD,
        k_neighbors=K_NEIGHBORS,
        radius_km=RADIUS_KM,
        min_k=MIN_K,
        max_k=MAX_K,
        rbf_sigmas=RBF_SIGMAS,
    )

    node_df["lat_norm"] = lat_norm
    node_df["lon_norm"] = lon_norm

    #merge normalized lat/lon back into panel data
    df = df.merge(
        node_df[["lat_norm", "lon_norm"]].reset_index(),
        on=NODE_COL,
        how="left",
    )

    #ensure lat_norm/lon_norm are included once in feature list
    for cname in ["lat_norm", "lon_norm"]:
        if cname not in FEATURE_COLUMNS:
            FEATURE_COLUMNS.append(cname)

    return df, node_df, node_index_map, pos_tensor, edge_index, edge_attr


In [ ]:
def build_dataset_from_panel(df_split, node_index_map, edge_index, edge_attr):
    data_list = []

    df_split["node_idx"] = df_split[NODE_COL].map(node_index_map)
    unique_times = df_split[TIME_COL].sort_values().unique()

    num_nodes = len(node_index_map)
    print("Nodes:", num_nodes, "Timesteps:", len(unique_times))

    for t in unique_times:
        df_t = df_split[df_split[TIME_COL] == t].sort_values("node_idx")
        if len(df_t) != num_nodes:
            continue

        x = torch.tensor(df_t[FEATURE_COLUMNS].values, dtype=torch.float32)
        y = torch.tensor(df_t[TARGET_COL].values, dtype=torch.float32)

        data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_attr)
        data_list.append(data)

    return data_list


In [ ]:
class DenseNet(nn.Module):
    def __init__(self, layers, activation):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(layers[i], layers[i+1])
            for i in range(len(layers)-1)
        ])
        self.activation = activation()

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        return self.layers[-1](x)


class KernelNN(nn.Module):
    """
    Graph Neural Operator:
      x_t → fc1 → NNConv(kernel(edge_attr))×depth → fc2 → y_t
    """
    def __init__(self, in_width, edge_feat_dim, width=96, ker_width=256, depth=5, out_width=1):
        super().__init__()
        self.depth = depth

        self.fc1 = nn.Linear(in_width, width)

        kernel = DenseNet([edge_feat_dim, ker_width, ker_width, width*width], nn.ReLU)
        self.conv1 = NNConv_old(width, width, kernel, aggr="mean")

        self.fc2 = nn.Linear(width, out_width)

    def forward(self, data):
        x = self.fc1(data.x)
        for _ in range(self.depth):
            x = F.relu(self.conv1(x, data.edge_index, data.edge_attr))
        return self.fc2(x).squeeze(-1)


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_mse = 0
    total_nodes = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            mse = F.mse_loss(out, batch.y, reduction="sum")
            total_mse += mse.item()
            total_nodes += batch.y.numel()
    return total_mse / total_nodes


def train_one_experiment(config=None):
    if config is None:
        config = BASE_CONFIG

    df, node_df, node_index_map, pos, edge_index, edge_attr = load_and_preprocess_panel(CSV_PATH)

    #Create 4 SCENARIO WEEKS (never used in training/validation/test)
    def week_mask(df_local, start, end):
        return (df_local[TIME_COL] >= start) & (df_local[TIME_COL] < end)

    winter_mask  = week_mask(df, "2025-01-10", "2025-01-17")
    spring_mask  = week_mask(df, "2025-04-10", "2025-04-17")
    summer_mask  = week_mask(df, "2025-07-10", "2025-07-17")
    autumn_mask  = week_mask(df, "2025-10-10", "2025-10-17")

    scenario_mask = winter_mask | spring_mask | summer_mask | autumn_mask

    df_scenarios = df[scenario_mask].copy()
    df_rest      = df[~scenario_mask].copy()

    print("Scenario rows:", len(df_scenarios), "| Train/Val/Test rows:", len(df_rest))

    #RANDOM TIME-LEVEL SPLIT OVER FULL YEAR (excluding scenario weeks)
    unique_times = df_rest[TIME_COL].sort_values().unique()

    rng = np.random.RandomState(0)
    perm = rng.permutation(len(unique_times))
    unique_times = unique_times[perm]

    n = len(unique_times)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    n_train = max(1, n_train)
    n_val   = max(1, n_val)

    train_times = unique_times[:n_train]
    val_times   = unique_times[n_train:n_train + n_val]
    test_times  = unique_times[n_train + n_val:]

    df_train = df_rest[df_rest[TIME_COL].isin(train_times)].copy()
    df_val   = df_rest[df_rest[TIME_COL].isin(val_times)].copy()
    df_test  = df_rest[df_rest[TIME_COL].isin(test_times)].copy()

    print("Random time split (excluding scenario weeks):")
    print("  Train timesteps:", len(np.unique(df_train[TIME_COL])))
    print("  Val timesteps:  ", len(np.unique(df_val[TIME_COL])))
    print("  Test timesteps: ", len(np.unique(df_test[TIME_COL])))
    print("  Rows train/val/test:", len(df_train), len(df_val), len(df_test))

    #compute per-node normalization STATS on TRAIN ONLY
    stats = (
        df_train.groupby(NODE_COL)[ORIGINAL_TARGET_COL]
        .agg(["mean", "std"])
        .rename(columns={"mean": "rl_mean", "std": "rl_std"})
    )
    stats["rl_std"].replace(0, 1.0, inplace=True)

    def apply_norm(split):
        split = split.merge(stats, left_on=NODE_COL, right_index=True, how="left")

        split["residual_load_norm"] = (
            (split[ORIGINAL_TARGET_COL] - split["rl_mean"]) / split["rl_std"]
        )

        split[TARGET_COL] = split["residual_load_norm"]

        cols_required = FEATURE_COLUMNS + [TARGET_COL]
        before = len(split)
        split = split.dropna(subset=cols_required)
        after = len(split)
        print(f"Dropped {before - after} rows in this split due to NaNs.")
        return split

    df_train = apply_norm(df_train)
    df_val   = apply_norm(df_val)
    df_test  = apply_norm(df_test)

    train_data = build_dataset_from_panel(df_train, node_index_map, edge_index, edge_attr)
    val_data   = build_dataset_from_panel(df_val,   node_index_map, edge_index, edge_attr)
    test_data  = build_dataset_from_panel(df_test,  node_index_map, edge_index, edge_attr)

    train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)

    in_width = len(FEATURE_COLUMNS)
    edge_feat_dim = edge_attr.size(-1)

    model = KernelNN(
        in_width=in_width,
        edge_feat_dim=edge_feat_dim,
        width=config["hidden_width"],
        ker_width=config["kernel_width"],
        depth=config["depth"],
        out_width=1,
    ).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val = float("inf")
    best_state = None
    train_losses = []
    val_losses = []

    for ep in range(config["num_epochs"]):
        model.train()
        total_mse = 0.0
        total_nodes = 0

        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch)
            loss = F.mse_loss(out, batch.y, reduction="mean")
            loss.backward()
            opt.step()

            total_mse += loss.item() * batch.y.numel()
            total_nodes += batch.y.numel()

        train_mse = total_mse / total_nodes
        val_mse = evaluate(model, val_loader)

        train_losses.append(train_mse)
        val_losses.append(val_mse)

        if ep % 5 == 0 or ep == config["num_epochs"] - 1:
            print(f"Epoch {ep:03d} | train MSE {train_mse:.4e} | val MSE {val_mse:.4e}")

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_mse = evaluate(model, test_loader)

    print(f"Best val MSE = {best_val:.4e}")
    print(f"Test MSE     = {test_mse:.4e}")

    return best_val, test_mse, model, train_losses, val_losses, test_loader, test_data, df_scenarios, stats


In [ ]:
val_mse, test_mse, model, train_losses, val_losses, test_loader, test_data, df_scenarios, stats = train_one_experiment()


Scenario rows: 13440 | Train/Val/Test rows: 132480
Random time split (excluding scenario weeks):
  Train timesteps: 4636
  Val timesteps:   993
  Test timesteps:  995
  Rows train/val/test: 92720 19860 19900
Dropped 11 rows in this split due to NaNs.
Dropped 0 rows in this split due to NaNs.
Dropped 0 rows in this split due to NaNs.
Nodes: 20 Timesteps: 4636


/tmp/ipython-input-487640801.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  stats["rl_std"].replace(0, 1.0, inplace=True)


Nodes: 20 Timesteps: 993
Nodes: 20 Timesteps: 995


/tmp/ipython-input-487640801.py:96: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
/tmp/ipython-input-487640801.py:97: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
/tmp/ipython-input-487640801.py:98: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)


Epoch 000 | train MSE 7.2508e-01 | val MSE 4.7246e-01
Epoch 005 | train MSE 2.2411e-01 | val MSE 1.9760e-01
Epoch 010 | train MSE 1.5500e-01 | val MSE 1.6127e-01
Epoch 015 | train MSE 1.1629e-01 | val MSE 1.2614e-01
Epoch 020 | train MSE 9.6512e-02 | val MSE 1.1645e-01
Epoch 025 | train MSE 8.0857e-02 | val MSE 8.6492e-02
Epoch 030 | train MSE 6.9299e-02 | val MSE 8.3287e-02
Epoch 035 | train MSE 5.6079e-02 | val MSE 8.1355e-02
Epoch 040 | train MSE 5.1106e-02 | val MSE 6.4187e-02
Epoch 045 | train MSE 4.5388e-02 | val MSE 6.1246e-02
Epoch 050 | train MSE 3.9834e-02 | val MSE 5.4305e-02
Epoch 055 | train MSE 3.5377e-02 | val MSE 4.7176e-02
Epoch 060 | train MSE 3.4094e-02 | val MSE 5.6728e-02
Epoch 065 | train MSE 3.1041e-02 | val MSE 4.5889e-02
Epoch 070 | train MSE 3.1048e-02 | val MSE 4.2396e-02
Epoch 075 | train MSE 2.7943e-02 | val MSE 5.7036e-02
Epoch 080 | train MSE 2.2085e-02 | val MSE 4.4102e-02
Epoch 085 | train MSE 2.6741e-02 | val MSE 4.1992e-02
Epoch 090 | train MSE 1.9356

In [ ]:
#save full artifacts+download
import os
import glob
import torch
import numpy as np
from google.colab import files

SAVE_DIR = "/content/gno_run_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

#recreate deterministic graphs
df_raw, node_df, node_index_map, pos, edge_index, edge_attr = load_and_preprocess_panel(CSV_PATH)

#rebuild same val/test sets + 4 scenario weeks
df = df_raw.copy()

def week_mask(df_local, start, end):
    return (df_local[TIME_COL] >= start) & (df_local[TIME_COL] < end)

winter_mask  = week_mask(df, "2025-01-10", "2025-01-17")
spring_mask  = week_mask(df, "2025-04-10", "2025-04-17")
summer_mask  = week_mask(df, "2025-07-10", "2025-07-17")
autumn_mask  = week_mask(df, "2025-10-10", "2025-10-17")
scenario_mask = winter_mask | spring_mask | summer_mask | autumn_mask

df_scenarios = df[scenario_mask].copy()
df_rest      = df[~scenario_mask].copy()

unique_times = df_rest[TIME_COL].sort_values().unique()
rng = np.random.RandomState(0)
unique_times = unique_times[rng.permutation(len(unique_times))]

n = len(unique_times)
n_train = max(1, int(n * 0.70))
n_val   = max(1, int(n * 0.15))

train_times = unique_times[:n_train]
val_times   = unique_times[n_train:n_train + n_val]
test_times  = unique_times[n_train + n_val:]

df_train = df_rest[df_rest[TIME_COL].isin(train_times)].copy()
df_val   = df_rest[df_rest[TIME_COL].isin(val_times)].copy()
df_test  = df_rest[df_rest[TIME_COL].isin(test_times)].copy()

#apply same per-node normalization stats
def apply_norm_with_stats(split, stats_df):
    split = split.merge(stats_df, left_on=NODE_COL, right_index=True, how="left")
    split["residual_load_norm"] = (split[ORIGINAL_TARGET_COL] - split["rl_mean"]) / split["rl_std"]
    split[TARGET_COL] = split["residual_load_norm"]
    split = split.dropna(subset=FEATURE_COLUMNS + [TARGET_COL])
    return split

df_val_n   = apply_norm_with_stats(df_val, stats)
df_test_n  = apply_norm_with_stats(df_test, stats)
df_scen_n  = apply_norm_with_stats(df_scenarios, stats)

#buiild pyg notebooks
val_data      = build_dataset_from_panel(df_val_n,  node_index_map, edge_index, edge_attr)
test_data     = build_dataset_from_panel(df_test_n, node_index_map, edge_index, edge_attr)
scenario_data = build_dataset_from_panel(df_scen_n, node_index_map, edge_index, edge_attr)

#package into artifact
artifact = {
    #model
    "model_state_dict": model.state_dict(),
    "model_config": {
        "in_width": len(FEATURE_COLUMNS),
        "edge_feat_dim": edge_attr.size(-1),
        "hidden_width": BASE_CONFIG["hidden_width"],
        "kernel_width": BASE_CONFIG["kernel_width"],
        "depth": BASE_CONFIG["depth"],
        "out_width": 1,
    },

    #graph
    "edge_index": edge_index.cpu(),
    "edge_attr": edge_attr.cpu(),
    "pos": pos.cpu(),

    #bookkeeping
    "node_index_map": node_index_map,
    "node_ids": list(node_index_map.keys()),

    # contract
    "feature_columns": FEATURE_COLUMNS,
    "target_col": TARGET_COL,
    "original_target_col": ORIGINAL_TARGET_COL,
    "norm_stats": stats.reset_index(),  # NODE_COL, rl_mean, rl_std

    #store exact split timestamps to avoid mixing
    "train_times": train_times,
    "val_times": val_times,
    "test_times": test_times,

    # raw normalized splits
    "df_val": df_val_n.reset_index(drop=True),
    "df_test": df_test_n.reset_index(drop=True),
    "df_scenarios": df_scen_n.reset_index(drop=True),

    #PyG datasets
    "val_data": val_data,
    "test_data": test_data,
    "scenario_data": scenario_data,
}

SAVE_PATH = os.path.join(SAVE_DIR, "gno_artifacts.pt")
torch.save(artifact, SAVE_PATH)

print(f"Saved artifacts to: {SAVE_PATH}")
print("Folder contents:", os.listdir(SAVE_DIR))
print("Artifact size (MB):", os.path.getsize(SAVE_PATH) / 1024 / 1024)

#zip + dowonload
ZIP_PATH = "/content/gno_run_artifacts.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

!zip -r gno_run_artifacts.zip /content/gno_run_artifacts > /dev/null

print("Zipped to:", ZIP_PATH)
print("Zip size (MB):", os.path.getsize(ZIP_PATH) / 1024 / 1024)

files.download(ZIP_PATH)


Nodes: 20 Timesteps: 993
Nodes: 20 Timesteps: 995
Nodes: 20 Timesteps: 672
Saved artifacts to: /content/gno_run_artifacts/gno_artifacts.pt
Folder contents: ['gno_artifacts.pt']
Artifact size (MB): 30.493207931518555
Zipped to: /content/gno_run_artifacts.zip
Zip size (MB): 12.689892768859863


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>